# 01 — Treino ResNet50

Fine-tuning do ResNet50 (ImageNet) para classificação de doenças de soja.

**Estratégia:** warm-up com backbone congelado → fine-tuning completo
**Tracking:** Weights & Biases (`ze-praga-models`)

## 1. Setup

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q wandb huggingface_hub timm
    !git clone https://github.com/SEU_USUARIO/tcc-ze-praga-model-playground.git
    %cd tcc-ze-praga-model-playground
    # Carrega secrets do Colab
    from google.colab import userdata
    import os
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    os.environ['HF_TOKEN']      = userdata.get('HF_TOKEN')

In [ ]:
import os
import sys
from pathlib import Path

import torch
import wandb
import yaml

# Adiciona src/ ao path
sys.path.insert(0, str(Path('.').resolve()))
from src.dataset import create_dataloaders
from src.models.resnet50 import build_resnet50, freeze_backbone, unfreeze_backbone
from src.trainer import Trainer
from src.evaluate import evaluate_model, plot_confusion_matrix

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Configuração

In [ ]:
with open('configs/training_config.yaml') as f:
    base_cfg = yaml.safe_load(f)

# Override inline se necessário
cfg = {
    'data_dir':       '/content/drive/MyDrive/ze-praga-dataset',  # altere aqui
    'batch_size':     base_cfg['training']['batch_size'],
    'epochs':         base_cfg['training']['epochs'],
    'lr':             base_cfg['training']['learning_rate'],
    'weight_decay':   base_cfg['training']['weight_decay'],
    'patience':       base_cfg['training']['patience'],
    'warmup_epochs':  base_cfg['training']['warmup_epochs'],
    'image_size':     base_cfg['dataset']['image_size'],
    'num_classes':    base_cfg['dataset']['num_classes'],
    'wandb_project':  base_cfg['wandb']['project'],
    'model_name':     'resnet50',
}

print('Config:', cfg)

## 3. Dataloaders

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    data_dir=cfg['data_dir'],
    batch_size=cfg['batch_size'],
    image_size=cfg['image_size'],
)

## 4. Modelo

In [ ]:
model = build_resnet50(num_classes=cfg['num_classes'], pretrained=True)
total_params = sum(p.numel() for p in model.parameters())
print(f'Parâmetros totais: {total_params:,}')

## 5. W&B + Treino

In [ ]:
wandb.login()
run = wandb.init(
    project=cfg['wandb_project'],
    name=f"resnet50-run",
    config=cfg,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg['lr'],
    weight_decay=cfg['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5, verbose=True
)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=DEVICE,
    checkpoint_dir='checkpoints',
    use_wandb=True,
)

best_model = trainer.fit(
    epochs=cfg['epochs'],
    patience=cfg['patience'],
    warmup_epochs=cfg['warmup_epochs'],
    freeze_fn=freeze_backbone,
    unfreeze_fn=unfreeze_backbone,
    model_name=cfg['model_name'],
)

## 6. Avaliação no Validation Set

In [ ]:
results = evaluate_model(best_model, val_loader, device=DEVICE)
print(f"Val Accuracy: {results['accuracy']:.4f}")

fig = plot_confusion_matrix(
    results['y_true'], results['y_pred'],
    save_path='checkpoints/resnet50_val_confusion_matrix.png',
    log_wandb=True,
)
plt.show()

In [ ]:
wandb.finish()
print('Treino concluído. Checkpoint salvo em checkpoints/best_resnet50.pth')